In [ ]:
import torch
import numpy as np
from train_convlstm import (
    ConvLSTMPredictor, load_ndmi_series, compute_time_deltas,
    predict_full_frame, BLOCK_SIZE, DELTA_THRESH
)
from scipy.ndimage import sobel, uniform_filter
import matplotlib.pyplot as plt

# Load trained model
model = ConvLSTMPredictor(input_channels=2, hidden_dims=(32, 32))
model.load_state_dict(torch.load("model.pt", map_location="cpu", weights_only=True))
model.eval()
print("Model loaded")

# Load data and run inference
series = load_ndmi_series()
ndmi_frames = [s[0] for s in series]
valid_masks = [s[1] for s in series]
date_strings = [s[2] for s in series]
time_deltas = compute_time_deltas(date_strings)

predicted = predict_full_frame(model, ndmi_frames, valid_masks, time_deltas)
last_observed = ndmi_frames[-1]
last_valid = valid_masks[-1]
print(f"Inference done on {len(series)} frames, last date: {date_strings[-1]}")

In [ ]:
# Compute change and directional gradients
delta = predicted - last_observed
deforest = (delta < DELTA_THRESH) & last_valid
delta_masked = np.where(deforest, delta, 0.0)
delta_smooth = uniform_filter(delta_masked.astype(np.float64), size=15)
grad_x = sobel(delta_smooth, axis=1)
grad_y = sobel(delta_smooth, axis=0)

H, W = last_observed.shape
n_rows, n_cols = H // BLOCK_SIZE, W // BLOCK_SIZE

arrow_x = np.zeros((n_rows, n_cols))
arrow_y = np.zeros((n_rows, n_cols))
arrow_mag = np.zeros((n_rows, n_cols))
block_delta = np.zeros((n_rows, n_cols))

for i in range(n_rows):
    for j in range(n_cols):
        r0, r1 = i * BLOCK_SIZE, (i + 1) * BLOCK_SIZE
        c0, c1 = j * BLOCK_SIZE, (j + 1) * BLOCK_SIZE
        block = deforest[r0:r1, c0:c1]
        if block.sum() < BLOCK_SIZE:
            continue
        gx = grad_x[r0:r1, c0:c1].mean()
        gy = grad_y[r0:r1, c0:c1].mean()
        mag = np.sqrt(gx**2 + gy**2)
        if mag > 1e-6:
            arrow_x[i, j] = gx / mag
            arrow_y[i, j] = -gy / mag
            arrow_mag[i, j] = mag
        block_delta[i, j] = delta[r0:r1, c0:c1].mean()

# Plot three panels
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

im0 = axes[0].imshow(np.where(last_valid, last_observed, np.nan),
                     cmap="RdYlGn", vmin=-0.5, vmax=0.7)
axes[0].set_title(f"Last Observed NDMI \u2014 {date_strings[-1]}", fontsize=13)
axes[0].axis("off")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(np.where(last_valid, predicted, np.nan),
                     cmap="RdYlGn", vmin=-0.5, vmax=0.7)
axes[1].set_title("Predicted Next NDMI", fontsize=13)
axes[1].axis("off")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(np.where(last_valid, delta, np.nan),
                     cmap="RdBu", vmin=-0.3, vmax=0.3)
axes[2].set_title("Predicted NDMI Change + Spread Direction", fontsize=13)
axes[2].axis("off")
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04, label="\u0394NDMI")

active = arrow_mag > 0
if active.any():
    yi, xi = np.where(active)
    cx = xi * BLOCK_SIZE + BLOCK_SIZE // 2
    cy = yi * BLOCK_SIZE + BLOCK_SIZE // 2
    u = arrow_x[active]
    v = arrow_y[active]
    colors = -block_delta[active]
    colors = np.clip(colors / max(colors.max(), 1e-6), 0, 1)
    axes[2].quiver(cx, cy, u, v, colors, cmap="YlOrRd",
                   scale=25, width=0.003, headwidth=4,
                   clim=(0, 1), alpha=0.8)

fig.tight_layout()
plt.show()
print(f"Arrows show predicted deforestation spread direction (threshold: \u0394NDMI < {DELTA_THRESH})")